# PCA lab

Load a table from `data/`, fit the local SVD module, inspect variance and scores.

Put your own CSV or Excel file in `data/` and change `DATA_PATH` below.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from pca import PCA, load_table, numeric_columns
from pca.plots import loadings_frame, scatter_figure, scree_figure

DATA_PATH = ROOT / "data" / "iris.csv"
LABEL_COL = "species"  # set to None if you have no group column
N_COMPONENTS = 2
STANDARDIZE = True

DATA_PATH

## Load the table

Numeric columns become features. Rows with empty numeric cells are dropped.

In [ ]:
frame = load_table(DATA_PATH)
features = [c for c in numeric_columns(frame) if c != LABEL_COL]
work = frame[features + ([LABEL_COL] if LABEL_COL else [])].dropna(subset=features)
X = work[features].to_numpy()
print(f"{DATA_PATH.name}: {X.shape[0]} rows, {len(features)} features")
print(features)
work.head()

## Fit PCA

Center each column. If `STANDARDIZE` is true, divide by the column standard deviation so units do not dominate. Then take the top right singular vectors.

In [ ]:
model = PCA(n_components=N_COMPONENTS, standardize=STANDARDIZE)
scores = model.fit_transform(X)

for i, (ratio, cumulative) in enumerate(
    zip(model.explained_variance_ratio_, model.cumulative_variance_ratio()), start=1
):
    print(f"PC{i}: {ratio:.3f} of variance  (cumulative {cumulative:.3f})")

pd.DataFrame(scores, columns=[f"PC{i+1}" for i in range(scores.shape[1])]).head()

## Scree and scatter

In [ ]:
scree_figure(model)

In [ ]:
labels = work[LABEL_COL] if LABEL_COL else None
scatter_figure(scores, labels)

## Loadings

Each row is an original column. Large absolute values mean that column leans hard on that component.

In [ ]:
loadings_frame(model, features).round(3)

## Save scores next to your data file

In [ ]:
out = DATA_PATH.with_name(DATA_PATH.stem + "_scores.csv")
score_frame = pd.DataFrame(scores, index=work.index, columns=[f"PC{i+1}" for i in range(scores.shape[1])])
if LABEL_COL:
    score_frame[LABEL_COL] = work[LABEL_COL].values
score_frame.to_csv(out, index=False)
print(f"wrote {out}")